In [2]:
pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 14.2 MB/s eta 0:00:00


In [3]:
import numpy as np
import skfuzzy as fuzz

In [4]:
temp_universe = np.arange(10, 41, 1)
hum_universe = np.arange(20, 101, 1)
stage_universe = np.arange(1, 4, 1)

In [5]:
# temperature MF
temp_very_low = fuzz.trapmf(temp_universe, [10, 10, 14, 16])
temp_low = fuzz.trimf(temp_universe, [14, 18, 22])
temp_optimal = fuzz.trapmf(temp_universe, [18, 22, 26, 28])
temp_high = fuzz.trimf(temp_universe, [26, 30, 34])
temp_very_high = fuzz.trapmf(temp_universe, [32, 35, 40, 40])

In [6]:
# humidity MF
hum_very_dry = fuzz.trapmf(hum_universe, [20, 20, 30, 35])
hum_dry = fuzz.trimf(hum_universe, [30, 40, 50])
hum_optimal = fuzz.trapmf(hum_universe, [45, 55, 65, 70])
hum_humid = fuzz.trimf(hum_universe, [65, 75, 85])
hum_very_humid = fuzz.trapmf(hum_universe, [80, 85, 100, 100])

In [7]:
# growth stage MF
stage_seedling = fuzz.trimf(stage_universe, [1, 1, 2])
stage_vegetative = fuzz.trimf(stage_universe, [1, 2, 3])
stage_flowering = fuzz.trimf(stage_universe, [2, 3, 3])

In [8]:
# fuzzify inputs
def fuzzify_inputs(temp, hum, stage):
    return {
        'temp_very_low': fuzz.interp_membership(temp_universe, temp_very_low, temp),
        'temp_low': fuzz.interp_membership(temp_universe, temp_low, temp),
        'temp_optimal': fuzz.interp_membership(temp_universe, temp_optimal, temp),
        'temp_high': fuzz.interp_membership(temp_universe, temp_high, temp),
        'temp_very_high': fuzz.interp_membership(temp_universe, temp_very_high, temp),

        'hum_very_dry': fuzz.interp_membership(hum_universe, hum_very_dry, hum),
        'hum_dry': fuzz.interp_membership(hum_universe, hum_dry, hum),
        'hum_optimal': fuzz.interp_membership(hum_universe, hum_optimal, hum),
        'hum_humid': fuzz.interp_membership(hum_universe, hum_humid, hum),
        'hum_very_humid': fuzz.interp_membership(hum_universe, hum_very_humid, hum),

        'stage_seedling': fuzz.interp_membership(stage_universe, stage_seedling, stage),
        'stage_vegetative': fuzz.interp_membership(stage_universe, stage_vegetative, stage),
        'stage_flowering': fuzz.interp_membership(stage_universe, stage_flowering, stage)
    }

In [9]:
# sugeno rule base
# temperature-humidity
sugeno_rules = [
    ('temp_very_low', 'hum_very_dry', 10, 60),
    ('temp_very_low', 'hum_optimal', 10, 20),
    ('temp_very_low', 'hum_very_humid', 15, 0),

    ('temp_low', 'hum_dry', 20, 50),
    ('temp_low', 'hum_optimal', 20, 20),
    ('temp_low', 'hum_humid', 25, 0),

    ('temp_optimal', 'hum_very_dry', 30, 70),
    ('temp_optimal', 'hum_dry', 30, 50),
    ('temp_optimal', 'hum_optimal', 25, 20),
    ('temp_optimal', 'hum_humid', 40, 0),

    ('temp_high', 'hum_very_dry', 70, 70),
    ('temp_high', 'hum_dry', 65, 50),
    ('temp_high', 'hum_optimal', 60, 20),

    ('temp_very_high', 'hum_dry', 90, 60),
    ('temp_very_high', 'hum_very_humid', 95, 0)
]

# growth stage
stage_adjustments = {
    'stage_seedling': (0.8, 1.2),
    'stage_vegetative': (1.0, 1.0),
    'stage_flowering': (1.1, 1.3)
}

In [10]:
# sugeno inference and output calculation
def sugeno_controller(temp, hum, stage):
    μ = fuzzify_inputs(temp, hum, stage)

    fan_num = 0
    mist_num = 0
    denom = 0

    # Core rules
    for t_key, h_key, fan_c, mist_c in sugeno_rules:
        firing = min(μ[t_key], μ[h_key])
        fan_num += firing * fan_c
        mist_num += firing * mist_c
        denom += firing

    # Avoid division by zero
    if denom == 0:
        fan = 0
        mist = 0
    else:
        fan = fan_num / denom
        mist = mist_num / denom

    # Growth stage adaptation
    if μ['stage_seedling'] > 0:
        fan *= stage_adjustments['stage_seedling'][0]
        mist *= stage_adjustments['stage_seedling'][1]
    elif μ['stage_flowering'] > 0:
        fan *= stage_adjustments['stage_flowering'][0]
        mist *= stage_adjustments['stage_flowering'][1]

    return fan, mist

In [11]:
# test sugeno controller
fan_out, mist_out = sugeno_controller(
    temp=30,
    hum=40,
    stage=1  # Seedling
)

print("Sugeno Fan Power:", fan_out)
print("Sugeno Misting:", mist_out)

Sugeno Fan Power: 52.0
Sugeno Misting: 60.0


In [12]:
# adaption function
def apply_adaptation(fan, mist, stage):
    """
    Adaptive scaling based on plant growth stage
    """

    if stage == 1:  # Seedling
        fan *= 0.8
        mist *= 1.2

    elif stage == 2:  # Vegetative
        fan *= 1.0
        mist *= 1.0

    elif stage == 3:  # Flowering
        fan *= 1.1
        mist *= 1.3

    # Ensure outputs stay within bounds
    fan = min(max(fan, 0), 100)
    mist = min(max(mist, 0), 100)

    return fan, mist

In [13]:
fan_raw, mist_raw = sugeno_controller(
    temp=30,
    hum=40,
    stage=3  # Flowering
)

fan_adapted, mist_adapted = apply_adaptation(
    fan_raw,
    mist_raw,
    stage=3
)

print("Adapted Sugeno Fan:", round(fan_adapted, 2))
print("Adapted Sugeno Mist:", round(mist_adapted, 2))

Adapted Sugeno Fan: 78.65
Adapted Sugeno Mist: 84.5
